# Herança de modelos

Como discutido na aula, herança de modelos pode ser usada para dividir os modelos em partes separadas — possivelmente para reaproveitamento.

Um uso principal da herança é ter um **modelo base customizado e padronizado**.

Por exemplo, suponha que queremos que todos os nossos modelos:
- ignorem campos extras;
- usem aliases gerados automaticamente;
- aceitem preenchimento tanto pelo nome quanto pelo alias.

Poderíamos definir essa configuração em cada modelo, mas isso não é o ideal.

Em vez disso, criamos um modelo base customizado (sem campos, só com a configuração):

In [1]:
from pydantic import BaseModel, ConfigDict
from pydantic.alias_generators import to_camel

In [2]:
class CustomBaseModel(BaseModel):
    model_config = ConfigDict(
        extra="ignore",
        alias_generator=to_camel, 
        populate_by_name=True
    )

Voltemos ao último exemplo de composição, agora usando o nosso modelo base:

In [3]:
from typing import Annotated

from pydantic import AfterValidator, EmailStr, Field, PastDate


SortedStringList = Annotated[list[str], AfterValidator(lambda value: sorted(value, key=str.casefold))]

class ContactInfo(CustomBaseModel):
    email: EmailStr | None = None

class PlaceInfo(CustomBaseModel):
    city: str
    country: str
    
class PlaceDateInfo(CustomBaseModel):
    date_: PastDate = Field(alias="date")
    place: PlaceInfo
    
class PersonalInfo(CustomBaseModel):
    nationality: str
    born: PlaceDateInfo

class Person(CustomBaseModel):
    first_name: str
    last_name: str
    contact_info: ContactInfo
    personal_info: PersonalInfo
    notable_students: SortedStringList = []

In [4]:
json_data = """
{
    "firstName": "David",
    "lastName": "Hilbert",
    "contactInfo": {
        "email": "d.hilbert@spectral-theory.com",
        "homePhone": {
            "countryCode": 49,
            "areaCode": 551,
            "localPhoneNumber": 123456789
        }
    },
    "personalInfo": {
        "nationality": "German",
        "born": {
            "date": "1862-01-23",
            "place": {
                "city": "Konigsberg",
                "country": "Prussia"
            }
        },
        "died": {
            "date": "1943-02-14",
            "place": {
                "city": "Gottingen",
                "country": "Germany"
            }
        }
    },
    "awards": ["Lobachevsky Prize", "Bolyai Prize", "ForMemRS"],
    "notableStudents": ["von Neumann", "Weyl", "Courant", "Zermelo"]
}
"""

In [5]:
p = Person.model_validate_json(json_data)

In [6]:
print(f"p.model_dump_json(by_alias=True, indent=2):\n{p.model_dump_json(by_alias=True, indent=2)}")

p.model_dump_json(by_alias=True, indent=2):
{
  "firstName": "David",
  "lastName": "Hilbert",
  "contactInfo": {
    "email": "d.hilbert@spectral-theory.com"
  },
  "personalInfo": {
    "nationality": "German",
    "born": {
      "date": "1862-01-23",
      "place": {
        "city": "Konigsberg",
        "country": "Prussia"
      }
    }
  },
  "notableStudents": [
    "Courant",
    "von Neumann",
    "Weyl",
    "Zermelo"
  ]
}


Outro uso da herança é quando queremos que todos os modelos tenham certos campos.

Por exemplo, numa API REST, podemos querer que toda resposta inclua informações básicas sobre a requisição: um ID único, a data e hora da requisição e quanto tempo ela levou.

Podemos usar herança para isso. Um aviso, porém: não caia na tentação da **herança múltipla** (modelos que herdam de vários modelos). A não ser que você domine herança múltipla e saiba que o uso pretendido é suportado pelo Pydantic, vai ter problemas. (Exemplo do instrutor: ele usava herança múltipla no Pydantic V1 para amarrar um modelo Pydantic a uma tabela do DynamoDB com todo o CRUD; a implementação quebrou por completo na migração para o V2, o refactor foi doloroso e ele acabou usando composição.)

Mas a herança **simples** funciona muito bem num cenário como o descrito acima.

Uma abordagem possível:

Começamos com uma base customizada sem campos — queremos a mesma configuração para todos os modelos, mas nem todo modelo da aplicação precisa dos campos padrão de informação da requisição.

Então mantemos essas coisas separadas.

Continuamos com o nosso modelo base:

In [7]:
class CustomBaseModel(BaseModel):
    model_config = ConfigDict(
        extra="ignore",
        alias_generator=to_camel, 
        populate_by_name=True
    )

Em seguida, criamos modelos para as informações da requisição. Reaproveito os tipos `Annotated` de datas e a serialização JSON de datetime que já implementamos — nada de novo, já vimos isso em seções anteriores.

In [8]:
from datetime import datetime
from typing import Any

import pytz
from dateutil.parser import parse
from pydantic import AfterValidator, BeforeValidator, FieldSerializationInfo, field_serializer, PlainSerializer


def make_utc(dt: datetime) -> datetime:
    if dt.tzinfo is None:
        dt = pytz.utc.localize(dt)
    else:
        dt = dt.astimezone(pytz.utc)
    return dt
    
def parse_datetime(value: Any):
    if isinstance(value, str):
        try:
            return parse(value)
        except Exception as ex:
            raise ValueError(str(ex))
    return value

def dt_serializer(dt, info: FieldSerializationInfo) -> datetime | str:
    if info.mode_is_json():
        return dt.strftime("%Y-%m-%dT%H:%M:%SZ")
    return dt
    
DateTimeUTC = Annotated[
    datetime, 
    BeforeValidator(parse_datetime), 
    AfterValidator(make_utc), 
    PlainSerializer(dt_serializer, when_used="unless-none")
]

Agora o modelo base para as respostas:

In [9]:
from uuid import uuid4

class RequestInfo(CustomBaseModel):
    query_id: uuid4 = Field(default_factory=uuid4)
    execution_dt: DateTimeUTC = Field(default_factory=lambda: datetime.now(pytz.utc))
    elapsed_time_secs: float

class ResponseBaseModel(CustomBaseModel):
    request_info: RequestInfo

> 📝 **Nota de revisão — bug:** `query_id: uuid4` usa a **função** `uuid4` como anotação de tipo, não um tipo de UUID. O default funciona (é o `default_factory` que gera o valor), mas qualquer valor **vindo de fora** é validado como "argumentos para chamar `uuid4`", e o schema JSON vira um objeto vazio. O tipo certo é `UUID4` (do Pydantic) ou `UUID`:

In [10]:
from pydantic import ValidationError

print(f"default gerado: {RequestInfo(elapsed_time_secs=1).query_id!r}")
try:
    RequestInfo(query_id="c4e60f4a-3c7f-4da5-9b3f-07aee50b23e7", elapsed_time_secs=1)
except ValidationError as ex:
    print(f"UUID válido rejeitado: {ex.errors()[0]['msg']}")
print(f"schema de queryId: {RequestInfo.model_json_schema()['properties']['queryId']}")

default gerado: UUID('1b4282c9-9311-4feb-80d7-4ec78f3bda42')
UUID válido rejeitado: Arguments must be a tuple, list or a dictionary
schema de queryId: {'additionalProperties': False, 'properties': {}, 'title': 'Queryid', 'type': 'object'}


Corrigindo:

In [11]:
from pydantic import UUID4

class RequestInfo(CustomBaseModel):
    query_id: UUID4 = Field(default_factory=uuid4)          # corrigido: UUID4 (tipo), uuid4 (fábrica)
    execution_dt: DateTimeUTC = Field(default_factory=lambda: datetime.now(pytz.utc))
    elapsed_time_secs: float

class ResponseBaseModel(CustomBaseModel):
    request_info: RequestInfo

print(f"UUID válido aceito: {RequestInfo(query_id='c4e60f4a-3c7f-4da5-9b3f-07aee50b23e7', elapsed_time_secs=1).query_id!r}")
print(f"schema de queryId: {RequestInfo.model_json_schema()['properties']['queryId']}")

UUID válido aceito: UUID('c4e60f4a-3c7f-4da5-9b3f-07aee50b23e7')
schema de queryId: {'format': 'uuid4', 'title': 'Queryid', 'type': 'string'}


`RequestInfo` e `ResponseBaseModel` herdam de `CustomBaseModel`, então têm a nossa configuração padrão.

E agora podemos usar `ResponseBaseModel` como base de todos os modelos de resposta da API.

In [12]:
class Users(ResponseBaseModel):
    users: list[str] = []

In [13]:
users = Users(request_info=RequestInfo(elapsed_time_secs=3.14), users=["Athos", "Porthos", "Aramis"])

In [14]:
print(f"users.model_dump_json(by_alias=True, indent=2):\n{users.model_dump_json(by_alias=True, indent=2)}")

users.model_dump_json(by_alias=True, indent=2):
{
  "requestInfo": {
    "queryId": "236df672-2a0e-44c9-9a8a-a0cf70021dd9",
    "executionDt": "2026-11-11T11:47:52Z",
    "elapsedTimeSecs": 3.14
  },
  "users": [
    "Athos",
    "Porthos",
    "Aramis"
  ]
}


> 📝 **Complemento — como a configuração é herdada:** o `model_config` de uma subclasse é **mesclado** com o da classe-mãe (as chaves da subclasse sobrescrevem; as demais são herdadas). Campos também são herdados, e a subclasse pode redefinir um campo.

In [15]:
class StrictUsers(Users):
    model_config = ConfigDict(extra="forbid")                 # só muda 'extra'

print(f"Users.model_config['extra']: {Users.model_config['extra']}")
print(f"StrictUsers.model_config['extra']: {StrictUsers.model_config['extra']}")
print(f"StrictUsers herdou o alias_generator: {StrictUsers.model_config['alias_generator'] is to_camel}")
print(f"StrictUsers herdou o populate_by_name: {StrictUsers.model_config['populate_by_name']}")
print(f"campos de StrictUsers: {list(StrictUsers.model_fields)}")
try:
    StrictUsers(requestInfo={"elapsedTimeSecs": 1}, users=[], page=2)
except ValidationError as ex:
    print(f"StrictUsers com extra: {ex.errors()[0]['type']} em {ex.errors()[0]['loc']}")

Users.model_config['extra']: ignore
StrictUsers.model_config['extra']: forbid
StrictUsers herdou o alias_generator: True
StrictUsers herdou o populate_by_name: True
campos de StrictUsers: ['request_info', 'users']
StrictUsers com extra: extra_forbidden em ('page',)
